# CE541E08 — Introduction to Python and MATLAB Programming
## Unit 2 · Day 15 — while Loop — Iteration Until a Condition is Met

---

| | |
|---|---|
| **Course** | CE541E08 — Introduction to Python and MATLAB Programming |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 2 — Decision Structures & Looping |
| **Session** | Day 15 of 45 |
| **CO** | CO2 — Use looping and conditional constructs to build programs |
| **Topics today** | while loop · reservoir filling · iterative solvers · convergence · Manning's back-calculation |

---
> Run each grey code cell with **Shift + Enter**. Read the explanation above each cell first.
> Instructor notes are marked `--- INSTRUCTOR NOTE ---`.
---

In [ ]:
# STUDENT HEADER — fill in before starting
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
github_repo  = "https://github.com/your-username/CE541E08-2026"
session      = "Day 15 — Unit 2"
print("╔══════════════════════════════════════════╗")
print("║         CE541E08 — Student Record         ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Name    : {student_name:<31}║")
print(f"║  Roll    : {roll_number:<31}║")
print(f"║  Session : {session:<31}║")
print("╚══════════════════════════════════════════╝")

## Section 1 — The while loop

```python
while condition:
    # runs as long as condition is True
    do_something()
    # must eventually make condition False, or loop runs forever
```

**for vs while:**
- `for` — you know in advance how many iterations (30 days, 6 pipe sections)
- `while` — you repeat until something happens (reservoir fills, solver converges)

**Critical rule:** Inside a while loop, something must change that will eventually make the condition False. Otherwise the loop runs forever (infinite loop).

### ▶ Cell 1 — Reservoir filling simulation

In [ ]:
# ---------------------------------------------------------
# WHAT THIS CELL DOES:
# Simulates a reservoir filling day by day.
# Inflow comes from a monsoon hydrograph, outflow is
# a constant controlled release. Loop until full or
# monsoon ends.
# ---------------------------------------------------------

# Reservoir parameters
capacity_Mm3    = 50.0    # million m³
initial_storage = 8.0     # starting storage, Mm3
outflow_Mm3day  = 0.8     # constant release, Mm3/day

# Monsoon inflow (Mm3/day) — 20-day sequence
inflow = [1.2, 1.8, 3.4, 6.7, 12.3, 18.5, 22.1, 19.8, 15.4,
          11.2, 8.9, 6.7, 5.4, 4.2, 3.1, 2.8, 2.1, 1.8, 1.4, 1.1]

storage  = initial_storage
day      = 0
spill    = 0.0

print(f"{'Day':>4} {'Inflow':>8} {'Outflow':>9} {'Net':>8} {'Storage':>10} {'Status'}")
print("-" * 55)

while storage < capacity_Mm3 and day < len(inflow):
    Q_in    = inflow[day]
    net     = Q_in - outflow_Mm3day
    storage += net
    if storage > capacity_Mm3:
        spill    = storage - capacity_Mm3
        storage  = capacity_Mm3
        status   = f"FULL + {spill:.2f} Mm³ spill"
    else:
        spill  = 0
        status = f"{storage/capacity_Mm3*100:.1f}% full"
    print(f"{day+1:>4} {Q_in:>8.2f} {outflow_Mm3day:>9.2f} {net:>+8.2f} {storage:>10.2f} {status}")
    day += 1

print("-" * 55)
if storage >= capacity_Mm3:
    print(f"Reservoir FULL on Day {day}")
else:
    print(f"Monsoon ended. Final storage: {storage:.2f} Mm³ ({storage/capacity_Mm3*100:.1f}%)")

# --- INSTRUCTOR NOTE ---------------------------------------
# Two conditions: storage < capacity AND days remaining.
# The 'and' means both must be true to continue.
# Ask: "What if we increase outflow to 2.0 Mm³/day?"
# Answer: reservoir may not fill — change outflow_Mm3day = 2.0
# -----------------------------------------------------------

## Section 2 — while loop for iterative engineering solvers

### ▶ Cell 2 — Iterative Manning's n back-calculation

In [ ]:
# ---------------------------------------------------------
# WHAT THIS CELL DOES:
# Back-calculates Manning's n from observed velocity using
# Newton-style iteration — a real field calibration technique.
# ---------------------------------------------------------

import math

# Observed field data
V_observed = 1.45   # measured velocity, m/s
R          = 0.38   # hydraulic radius, m
S          = 0.002  # measured slope

# Target: find n such that V_manning(n) = V_observed
# V = (1/n) * R^(2/3) * S^0.5
# → n = (1/V) * R^(2/3) * S^0.5

# Analytical solution (exact)
n_exact = (1/V_observed) * R**(2/3) * S**0.5
print(f"Exact n (analytical): {n_exact:.5f}")

# Iterative solution (Newton's method — to demonstrate while loop)
n_guess    = 0.020     # initial guess
tolerance  = 0.000001  # convergence criterion
iteration  = 0
max_iter   = 100

print()
print(f"  Iterative solution:")
print(f"{'Iter':>5} {'n_guess':>10} {'V_calc':>10} {'Error':>12}")
print("-"*40)

while True:
    V_calc = (1/n_guess) * R**(2/3) * S**0.5
    error  = V_observed - V_calc
    print(f"{iteration:>5} {n_guess:>10.6f} {V_calc:>10.4f} {error:>12.6f}")

    if abs(error) < tolerance:
        break
    # Update: adjust n proportionally
    n_guess = n_guess * (V_calc / V_observed)
    iteration += 1
    if iteration > max_iter:
        print("Did not converge!")
        break

print()
print(f"  Converged after {iteration} iterations")
print("Calibrated n =", round(n_guess, 5))
print("Typical values: concrete=0.013, earth=0.025, grass=0.035")

# --- INSTRUCTOR NOTE ---------------------------------------
# while True with break is a common pattern for iterative solvers.
# The convergence criterion abs(error) < tolerance is the key.
# Ask: "Why not use while error > tolerance directly?"
# Answer: error might be negative - abs() handles both sides.
# -----------------------------------------------------------

### ▶ Cell 3 — Pipe network Hardy-Cross iteration (simplified)

In [ ]:
# ---------------------------------------------------------
# WHAT THIS CELL DOES:
# A simplified Hardy-Cross iteration for a 2-pipe loop.
# Adjusts flow until head loss balances around the loop.
# ---------------------------------------------------------

# Single loop: two pipes in parallel
# Pipe 1: L=200m, D=200mm  |  Pipe 2: L=300m, D=250mm
import math

f=0.018; g=9.81; Q_total=0.05   # total flow to split, m³/s

def head_loss(Q, L, D):
    # Darcy-Weisbach head loss for pipe.
    A = math.pi*(D/2)**2
    V = Q/A
    return f*(L/D)*(V**2/(2*g))

L1, D1 = 200, 0.200
L2, D2 = 300, 0.250

# Initial flow split (proportional to D²)
Q1 = Q_total * D1**2 / (D1**2 + D2**2)
Q2 = Q_total - Q1

iteration = 0
tolerance = 0.0001

print(f"{'Iter':>5} {'Q1(L/s)':>10} {'Q2(L/s)':>10} {'hf1(m)':>8} {'hf2(m)':>8} {'dQ':>10}")
print("-"*55)

while True:
    hf1 = head_loss(Q1, L1, D1)
    hf2 = head_loss(Q2, L2, D2)
    imbalance = hf1 - hf2

    # Head loss derivative (2*hf/Q)
    dQ = imbalance / (2*(hf1/Q1 + hf2/Q2))
    print(f"{iteration:>5} {Q1*1000:>10.3f} {Q2*1000:>10.3f} {hf1:>8.4f} {hf2:>8.4f} {dQ*1000:>10.4f}")

    if abs(imbalance) < tolerance:
        break
    Q1 -= dQ
    Q2 += dQ
    iteration += 1
    if iteration > 50:
        break

print()
print(f"  Balanced solution after {iteration} iterations:")
print(f"  Q1 = {Q1*1000:.3f} L/s  |  Q2 = {Q2*1000:.3f} L/s")
print(f"  hf1 = {head_loss(Q1,L1,D1):.4f} m  |  hf2 = {head_loss(Q2,L2,D2):.4f} m")

# --- INSTRUCTOR NOTE ---------------------------------------
# Hardy-Cross is the standard method for pipe network analysis.
# Unit 5 (MATLAB) builds on this for a full 6-pipe network.
# Ask: "Why must hf1 = hf2 at balance?"
# Answer: energy is conserved around the loop — Kirchhoff's law.
# -----------------------------------------------------------

---
## Day 15 Assignment

A **detention pond** receives stormwater from an urban catchment. Simulate the storage routing:

- Pond capacity: 8000 m³
- Initial storage: 500 m³
- Inflow (m³/hr): `[200, 450, 890, 1240, 980, 720, 540, 380, 250, 180, 120, 80]`
- Outflow rate: 300 m³/hr (constant outlet)

Using a while loop:
1. Route the flood through the pond hour by hour
2. Track when the pond reaches 80% capacity (early warning)
3. Track when/if the pond overflows
4. Print a routing table and summary

### ▶ Assignment cell

In [ ]:
inflow_m3hr = [200,450,890,1240,980,720,540,380,250,180,120,80]
capacity   = 8000.0
storage    = 500.0
outflow    = 300.0
warning_80 = capacity * 0.8
hour       = 0

print(f"{'Hr':>4} {'Inflow':>8} {'Out':>6} {'Storage':>10} {'%Full':>7} {'Note'}")
print("-"*50)

while hour < len(inflow_m3hr):
    Qin     = inflow_m3hr[hour]
    net     = Qin - outflow
    storage += net
    overflow = 0
    note     = ""

    if storage >= capacity:
        overflow = storage - capacity
        storage  = capacity
        note     = f"OVERFLOW {overflow:.0f} m³"
    elif storage >= warning_80:
        note     = "⚠ 80% warning"

    print(f"{hour+1:>4} {Qin:>8.0f} {outflow:>6.0f} {storage:>10.1f} {storage/capacity*100:>6.1f}% {note}")
    hour += 1

print()
print(f"  Final storage: {storage:.1f} m³ ({storage/capacity*100:.1f}%)")

---
## Before Day 16 — checklist
- [ ] Completed all assignment cells above
- [ ] Saved notebook to Google Drive
- [ ] Uploaded to GitHub: `Unit2_LoopsDecisions/CE541E08_U2_Day15.ipynb`
- [ ] Commit message: `Day 15 — break, continue, pass`

---
*CE541E08 · Civil Engineering · Christ University · 2026–27 · Dr. Arpan Pradhan · arpan.pradhan@christuniversity.in · 9439291900*